# Task 3.1.7: Evaluating Object Tracking Algorithms

In [1]:
# Importing required libraries, which include:
# OpenCV library used for the many tasks in computer vision, from loading images, to processing, detecting, shapes, tracking objects etc.:
import cv2
# Os is a library that provides functions for creating and removing a directory (folder), fetching its contents etc.:
import os
# Numpy is very useful computation library, here we can use it to compute means:
import numpy as np
# Re is a library that provides functions to work with regular expressions:
import re

In [2]:
# UTILS
def draw_bounding_box(image, bbox, color, text=None):
    """
    Draws a bounding box on the given image.

    Args:
        image (numpy.ndarray): The image to draw the bounding box on.
        bbox (tuple): The bounding box given in [x, y, w, h] format.
        color (tuple): The color of the bounding box in BGR format.
        text (str, optional): The text to display near the bounding box. Defaults to None.

    Returns:
        numpy.ndarray: The image with the bounding box drawn.
    """
    start_point = (int(bbox[0]), int(bbox[1]))
    end_point = (int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3]))

    if text is not None:
        image = cv2.putText(image, text, (int(start_point[0]), int(start_point[1]) - 5), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    return cv2.rectangle(image, start_point, end_point, color = color, thickness = 2)

In [4]:
# In this code snippet, we are loading the ground truth bounding boxes values for the video sequence.
# open the ground truth file in read mode
with open("./data/got10k/groundtruth.txt", mode="r", encoding="utf-8") as file:
    # read the content of the file
    data = file.read()
    # define a pattern to split the data into separate values
    split_pattern = ",|\n"
    # split the data into separate values using the pattern
    bounding_boxes = re.split(split_pattern, data)
    # convert the values to integers and store them in a numpy array
    bounding_boxes = np.array([int(float(elem)) for elem in bounding_boxes if elem != ""])
    # reshape the numpy array into a 2D array, where each row represents a bounding box
    bounding_boxes = bounding_boxes.reshape(-1, 4)


# create an empty dictionary to store file names and their corresponding bounding boxes
filenames_bboxes = {}
# initialize the index for bounding box list to 0
idx = 0
# walk through the directory of image files and their respective bounding boxes
# note: The "." means the current directory
for root, _, files in os.walk("./data/got10k/sequence_example"):
    # iterate through each file in the directory in ascending order
    for filename in sorted(files):
            # Store the file name and its bounding box coordinates in the dictionary
            filenames_bboxes[filename] = bounding_boxes[idx]
            idx += 1

In [11]:
def calculate_iou(gt_bbox, pred_bbox):
    """
    Calculate Intersection over Union (IoU) between two bounding boxes.
    """
    
    gt_x, gt_y, gt_w, gt_h = gt_bbox
    pred_x, pred_y, pred_w, pred_h = pred_bbox
    
    inter_x1 = max(gt_x, pred_x)
    inter_y1 = max(gt_y, pred_y)
    inter_x2 = min(gt_x + gt_w, pred_x + pred_w)
    inter_y2 = min(gt_y + gt_h, pred_y + pred_h)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)

    intersection_area = inter_w * inter_h
    
    gt_area = gt_w * gt_h
    pred_area = pred_w * pred_h
    
    union_area = gt_area + pred_area - intersection_area
    
    if union_area == 0:
        return 0, [0, 0, 0, 0]

    iou = intersection_area / union_area

    intersection_bbox = [inter_x1, inter_y1, inter_w, inter_h]

    return iou, intersection_bbox

In [17]:
def draw_bounding_boxes_IoU(image, gt_bbox, pred_bbox, intersection_bbox, iou):

    # predicted
    image = draw_bounding_box(image, pred_bbox, (0, 0, 255), "Predicted")
    
    # ground truth
    image = draw_bounding_box(image, gt_bbox, (255, 0, 0), "Ground Truth")

    # intersection
    image = draw_bounding_box(image, intersection_bbox, (0, 255, 0), "IoU Area")
    
    # iou
    text = f"IoU: {iou:.2%}"
    
    cv2.putText(
        image,
        text,
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    return image

In [18]:
# Define image path
image_path = "./data/got10k/sequence_example/00000001.jpg"
# Load the image
image = cv2.imread(image_path)
# Get ground truth bounding box
gt_bbox = filenames_bboxes["00000001.jpg"]
# Simulate predicted bounding box. 
# Try changing the values and observe changes in the IoU value.
pred_bbox = [200, 200, 400, 300]

# Calculate IoU and get intersection bounding box
iou, intersection_bbox = calculate_iou(gt_bbox, pred_bbox) 
image = draw_bounding_boxes_IoU(image, gt_bbox, pred_bbox, intersection_bbox, iou)
cv2.imshow("IoU", image)
if cv2.waitKey(0) == ord("q"):
    cv2.destroyAllWindows()

In [19]:
# TASK3: Calculate mean Intersection over Union (mIoU). Round up to 1 decimal.
# Hint: To calculate the mean Intersection over Union (mIoU) for a video sequence, compute it as the average of IoU scores from every frame within the sequence.
def calculate_mIoU(ious):
    return round(np.mean(ious) * 100, 1)

In [27]:
# TASK4: Calculate Success Rate (SR) in percentages. Round up to 1 decimal.
# Hint: SR is calculated as the number of successful frames divided by the number of all frames in a video sequence.
#       Whether a frame is successful or not is determined based on the predefined threshold. 
#       Common threshold values are 0.5 and 0.75.
def calculate_sr(ious, threshold):
    counter = 0
    for iou in ious:
        if iou*100 >= threshold:
            counter += 1
            
    return round((counter / len(ious)) * 100, 1)

In [ ]:
for tracking_method in ["tracker_boosting", "tracker_KCF"]:
    # Set value to threshold for determining TP/FP/FN and Success Rate. In percentages.
    threshold = 50

    # Initialize the tracker with the first frame and its bounding box coordinates
    frame = cv2.imread("./data/got10k/sequence_example/00000001.jpg")
    bbox = filenames_bboxes["00000001.jpg"]

    if (tracking_method == "tracker_boosting"):  
        tracker = cv2.legacy.TrackerBoosting_create()
        # tracker = cv2.TrackerKCF_create()
    else:
        tracker = cv2.TrackerKCF_create()
    tracker.init(frame, bbox)

    # Set the path to the directory containing the frames
    frame_dir = "./data/got10k/sequence_example/"
    # Get a list of all the frame filenames in the directory and sort them
    frame_filenames = sorted(os.listdir(frame_dir))

    # Create a list of IoU values
    ious = []

    # loop through the frames
    for filename in frame_filenames:
        # read the frame from file
        frame_path = os.path.join(frame_dir, filename)
        frame = cv2.imread(frame_path)

        # Update the tracker with the new frame
        ret, pred_bbox = tracker.update(frame)
        
        # If tracking was successful
        if ret:
            cv2.putText(frame, f"Method {tracking_method}", (int(frame.shape[0]/2), 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
            # Draw ground truth bounding box as red rectangle
            gt_bbox = filenames_bboxes[filename]

            # Calculate IoU and intersection bounding box
            iou, intersection_bbox = calculate_iou(gt_bbox, pred_bbox)
            
            # Append IoUs list
            ious.append(iou)

            # Visualize bounding boxes
            frame = draw_bounding_boxes_IoU(frame, gt_bbox, pred_bbox, intersection_bbox, iou)
            cv2.imshow("Video sequence", frame)
            if cv2.waitKey(1) == ord("q"):
                cv2.destroyAllWindows()
            
    # Close the window with the image
    cv2.destroyAllWindows()

    # Calculate mIoU
    mIoU = calculate_mIoU(ious)
    
    # Calculate Success Rate
    sr = calculate_sr(ious, threshold)

    # Print results
    print(f"For method {tracking_method}:\n mIoU: {mIoU}%. SR with threshold {threshold}%: {sr}%.")

For method tracker_boosting:
 mIoU: 56.2%. SR with threshold 50%: 58.3%.
For method tracker_KCF:
 mIoU: 54.6%. SR with threshold 50%: 53.3%.


: 